# Responses API
Responses API는 OpenAI의 최신 API 흐름이다.
기존 chat copletions API처럼 텍스트를 생성할 수 있고, 구조화 출력, 도구 출력, 멀티턴 대화 같은 기능을 더 일관된
방식으로 다룰 수 있다. 

In [2]:
# 공통 설정
# .env 파일에서 API Key와 기본 모델 명을 읽어온다
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY를 환경변수로 설정하세요")

client = OpenAI(api_key=api_key)
DEFAULT_MODEL = os.getenv("OPENAI_DEFAULT_MODEL","gpt-4.1-mini")

print("OpenAI client 준비 완료")
print("기본 모델 :", DEFAULT_MODEL)

OpenAI client 준비 완료
기본 모델 : gpt-4.1-mini


## 기본 호출
- 기존 chat completions의 system message는 intructions로 옮겨간다.
- user message(사용자 요청)은 input으로 옮겨간다.
- 응답 객체의 output_text로 접근하면 응답 텍스트를 확인할 수 있다.

In [4]:
response = client.responses.create(
    model=DEFAULT_MODEL,
    instructions="너는 초급 개발자에게 쉽게 설명하는 AI 강사이다.",
    input="Responses API의 핵심 특징 3개 bullet으로 설명해줘.",
    temperature=0.3
)

print(response.output_text)

Responses API의 핵심 특징 3가지를 쉽게 설명해줄게요!

- **빠른 응답 생성**: 사용자의 입력에 대해 즉시 적절한 답변을 만들어줘요.
- **다양한 형식 지원**: 텍스트, 이미지, 버튼 등 여러 형태로 응답을 제공할 수 있어요.
- **유연한 맞춤 설정**: 필요에 따라 응답 스타일이나 내용을 쉽게 조절할 수 있어요.

필요하면 더 자세히도 설명해줄게요!


## previous_response_id로 멀티턴 대화 만들기

In [3]:
first = client.responses.create(
    model=DEFAULT_MODEL,
    instructions="너는 초급자에게 비유를 잘 드는 AI 강사다.",
    input="RAG를 한 문자으로 설명해줘."
)

print(first.output_text)
print("response id:",first.id)

RAG를 한 문자로 비유하면, ‘🔍’ (돋보기) 같아요.

왜냐하면 RAG는 정보를 찾고(검색, Retrieval), 그 정보를 바탕으로 답변을 생성(Generation)하는 방식이거든요. 마치 돋보기가 필요한 부분을 확대해서 더 잘 보게 해 주는 것처럼요!
response id: resp_0aae02d98794ec5b0069fbe59ae34081939486c14412beb98a


In [4]:
second = client.responses.create(
    model=DEFAULT_MODEL,
    previous_response_id=first.id,
    input="방금 설명을 도서관 비유로 바꿔줘."
)

print(second.output_text)
print("response id:",second.id)

RAG를 도서관 비유로 설명하자면,  
"RAG는 도서관 사서와 같아요."

사용자가 질문하면,  
1) 사서가 먼저 도서관(외부 지식)에서 관련 책이나 자료를 찾아서 꺼내오고(검색, Retrieval),  
2) 그 자료를 바탕으로 질문에 맞는 답을 만들어서(생성, Generation) 알려주는 역할을 하거든요.

즉, RAG는 정보를 직접 외우지 않고 필요할 때마다 도서관에서 찾아서 정확한 답을 제공하는 똑똑한 사서 같은 시스템입니다!
response id: resp_0aae02d98794ec5b0069fbe6383f2081939455ead19fe9d4f8


## 반복 입력으로 대화 이어하기

In [5]:
print("종료하려면 exit을 입력하세요")

previous_response_id = None

while True:
    user_input = input("User : ")

    if user_input.strip().lower() == 'exit':
        print("채팅을 종료합니다.")
        break

    # 첫 요청에는 previous_response_id가 없으므로 두 번째 요청부터 전달한다.
    request_params = {
        "model" : DEFAULT_MODEL,
        "instructions" : "너는 Python 수업을 돕는 AI 튜터이다. 답변은 3문장 이내로 한다.",
        "input":user_input,
        "temperature":0.4
    }

    if previous_response_id is not None:
        request_params['previous_response_id']=previous_response_id

    response = client.responses.create(**request_params)

    print("assistant : ",response.output_text)

    # 다음 대화를 위해 이번 응답 id를 저장
    previous_response_id = response.id

종료하려면 exit을 입력하세요
assistant :  기초 문법부터 차근차근 배우고, 간단한 코드를 직접 작성해보세요. 문제를 많이 풀면서 실습하는 것이 중요합니다. 궁금한 점은 바로바로 질문하며 이해를 깊게 하세요.
assistant :  파이썬은 문법이 쉽고 다양한 분야에서 활용돼 입문자에게 적합해요. 데이터 분석, 인공지능, 웹 개발 등 여러 분야에서 수요가 높습니다. 배우면 실무와 취업에 큰 도움이 됩니다.
assistant :  파이썬은 문법이 쉽고 다양한 분야에서 활용돼 배우기 좋으며, 실무와 취업에 유리합니다. 공부할 때는 기초부터 차근차근 배우고, 직접 코드를 작성하며 문제를 많이 푸는 것이 중요합니다. 궁금한 점은 바로 질문해 이해를 깊게 하세요.
채팅을 종료합니다.
